<div dir="rtl">

# HM_09 - מעבדה: דיבוג קוד שנוצר ב-AI - התקלה שמאחורי ההזיה

**הקורס:** מודלי שפה וסוכנים ממוחשבים למסלול מערכות מידע (4261310) ולמסלול ידע ונתונים (תשפז_ב_4261110) · מאת ד״ר ולד לנדא וד״ר רן דובין · **מפגש 9:** דיבוג, ניתוח שורש ובדיקת רגרסיה

**שם:** _(כתבו כאן)_

---

## מה עושים במעבדה

במפגש 9 למדנו ש**דיבוג הוא שיטה** ולא ניחוש: משחזרים, מבודדים, משערים, בודקים, מתקנים, מאמתים ומונעים. ראינו שני סוגי תקלות בקוד שנכתב ב-AI: **תקלה רועשת** (יש Traceback, ולפעמים זו הזיה של פונקציה או חבילה) ו**תקלה שקטה** (אין שגיאה, והסכום פשוט לא נכון). במעבדה הזו תבנו בעצמכם את הכלים של המתקן: **קורא Traceback, בודק קיום API, בדיקות שפיות, מאתר שורות בעייתיות, ובדיקת רגרסיה** שמוכיחה שהתיקון באמת תיקן.

המשימה שמלווה אותנו: **דוח הכנסות רבעון 3 לפי ספק** שנכתב בעזרת AI. הדוח רץ בלי שגיאה ומחזיר **88,526.81**, אבל הנהלת החשבונות אומרת **177,597.89**. תמצאו למה, תתקנו, ותוסיפו בדיקה שלא תיתן לבאג לחזור בשקט.

| חלק | נושא | מה מממשים | זמן |
|---|---|---|---|
| א' | קוראים Traceback ובודקים שה-API קיים | `parse_traceback`, `api_exists`, `package_exists` | 8 דק' |
| ב' | משחזרים ומבודדים באג שקט | `count_nat`, `count_dup_ids`, `find_bad_rows`, מקרה מינימלי | 10 דק' |
| ג' | מתקנים, בדיקת רגרסיה, תחקיר | ארבעה תיקונים, `test_regression`, שורש ו-commit | 10 דק' |
| ד' | טקסונומיה של באגים | תיוג תשעה באגים (מילון) | 3 דק' |
| ה' | דיבוג ביחד עם AI (צעד אמיתי) | `build_debug_request`, `patch_smells`, הערכת האבחנה | 5 דק' |
| ו' | מסקנה | - | 3 דק' |

> **אם הזמן קצר:** חלקים א'-ג' הם הליבה (28 דק'). חלקים ד'-ה' קצרים - השלימו אותם בבית אם צריך. בונוס בסוף המחברת.

**איך זה מתחלק במפגש (שקף ״מפת המפגש״):** מעבדה א׳ = חלק א׳ (10 דק׳, כולל דיון), מעבדה ב׳ = חלק ב׳ (10 דק׳), מעבדה ג׳ = חלקים ג׳-ו׳ (18 דק׳; שלב 6 בחלק ג׳ והמסקנה אפשר בבית).

## איך עובדים

1. **מממשים** כל פונקציה שמסומנת `# TODO` - **בעצמכם, בלי כלי AI**. המטרה היא להבין איך מתקנים, לא לקבל קוד. (רק בחלק ה' משתמשים בכלי AI, כי הוא נושא הניסוי.)
2. **מריצים את תא הבדיקה** שאחרי כל פונקציה. כשהמימוש נכון יודפס `✓`; אחרת תופיע הודעה שמסבירה מה נכשל.
3. **לפני כל הרצה מפתיעה כותבים תחזית** בתא המיועד לכך. תחזית "שגויה" עם ניתוח טוב שווה יותר מתחזית "נכונה" בלי הסבר.
4. **עונים על שאלות הניתוח** בתאי ה"תשובה", במונחים של מה שלמדנו בהרצאה.

> **אל תיבהלו מהקוד.** כל פונקציה כאן היא 2-8 שורות, ובכל שלב מופיעים הסבר, דוגמה ורמז. אם נתקעתם: קראו שוב את ה-docstring (הטקסט בין שלוש המרכאות), ובקשו עזרה מבן הזוג.

## הגשה

בסוף: **Runtime ← Restart session and run all**, ודאו שתא הבדיקה המסכם בסוף המחברת מראה `✓` לכל הפונקציות, ואז **File ← Download ← Download .ipynb** והגישו את הקובץ.

</div>

In [ ]:
# תא הכנה - הריצו פעם אחת בתחילת העבודה
import linecache
import traceback
import importlib.util
import pandas as pd

print("✓ מוכן")

In [ ]:
#@title תא הכנה - הנתונים (אפשר לא לפתוח) { display-mode: "form" }
import base64, gzip, pathlib
_FILES = {
    'orders_il.csv': (
        '...'
    ),
    'products.csv': (
        'H4sIAOIO2WoC/21Z227cyBF9D5B/mA9gCPa9+3Etx+vE66xgOTDytKBHtDTQaDiYSxzn6/dUNfvG8ZMk6DR5qvrUlcfT/HjdXv7Y'
        'PXaH8XXqtuNleppPP7rrYXf5YzufL/G342m3nbrz9Xjc76YT4H/9y/0wiO7tdH7Z/Da+HjdvxvNu2/3+7RsBZR98Z3ulu4fBMFRW'
        '0Pv99ZyQVvdBdUKYXhuAPYNVDT7NCat971wXXD+U5+oK+nF32CWssr3vjOxNANQy1HT/mi/T13l+2fxiWr4u9E6Aheo94SXjbYOv'
        'SYOIBXyAmZmIa9GFtRQE1ro3hbVvwDVv03vThV7JTDt0v077zf102DxMl5a2kuQK63rnk+/E0MAbV7s+hE5I2UuR4aKFF9pK9zp0'
        'BlToXhyDZQNuaNve6c6L3hATwWjVPVzG4346taSNIuu86oXMUJ2hNWHnew03K9MPIt2KMAVayOL1QfKF8EMVI21GNsKAh0Xnht7m'
        'R7qood9PT+Nh9/8bvkPvbOdDH1z2ml+faGgPPXERvTPpwkW4OVCRH3qEB3zHVjJeDmt8bQLJjyRbIgYX+uV5R4oaT4+bj+PpZW2F'
        'lb31UeFkuOZT8ienakuk6g2ku8QmX5VUPztTjBGOZCNtP2RJSv2TI7U9zvWGqVmfVC/pmnfblx8bCpTzyhbbWxIyPFBeYtsDTbgq'
        'ujvre6My3K3gxQKEibKgo3sCLVb7Fl6zh2cFZTBHYl0kJUN3Px5h5t1+dwT7+X+tBXqgcEHuKxJRw82Rxgbbi8XwJRqVuD3QWIE3'
        'COGrjIawW59YGeJCF5DURLoGpbqPM/L/fNoglg6PrRUC6cd1eEXIKQIZvz1Q2wAXKWjDUD5MlMwaX2UgSt4QkspguwI3QaEoc3pH'
        'wZrwbvnn5u553EFz4yp/xrSB0JYyX4O/PVPbYHyPYA4mJjpONLDp9khVtSgnLRfH2tPDLb65BxZgQPJTKVKhrH8/vPnb3eZu/Lqf'
        'FiM+T9vnTsFLnUWW9skCLRssk2eoGDgF4OkiUccbGixYM9SZXpGmB6rj6bm6wTJjBkPMKG1Ox5uKzzXdl91p2k9nxMp8PbeUuV7h'
        'FTbXCm3X+ELbKS62VI98ulntbvCJuiX5Qpk+5xIcW4EL94Dci2BXopcqkwndh+lHzFY17dAbR2aqXPTNUJCFMPLlQKHKNZxvHEm0'
        '4LKPETeIUFwglb/oNyMLsJCk5khTTrUyxRky2Zfp63Z8rRmikBiuVCYLBxez4Bp+qgss/fQ0k1HFjfAx7r/IHI9eQIWZ4wQgyYeL'
        '8+Ci++f5MDX5Ir5WklJtbIMiOd9gK4bch6GO+nwpSAUNNvFE1cArUaiy9uzQICs3InQB5QK1GA5h3c/fuVYeXmqyUvdeMuWcTeD8'
        'ClvISs+NkouNUqSgGmjiCpYODxXMNb5f18BCFT0aWgLqGXOJt6Z7//bjPza/PI7HS67uUZkwDK415IlFmda26CqcbCxvhn4sF2Hd'
        'Cp0YK35wb21m4VtkoWwEaQHVw+QODCnvN8Dm4+ZhP03/bZKAYyBFdRaYG1bwQhpBgkYXnaPIyka/vkJXkoDzJEpsxsoVtkpe6KVQ'
        'NmT0RuShujf763SZ58vz5uE4jS+tuxEVaJtF4A6SrxGV9vZElXgdpVG51EnOSaj7PzmRLCBfKm5nc8ZDVbs9UMyABGCAq+cXjEp/'
        'H09fr4/nRtiKijynqEUqyMUJV1H2PXoeZHaVKyOOZVxWtKPpzYqoU36tHzKsCr1Ab0XiZ3FEnOju5m/fJlzG9Wlh+GF32T5PBy6y'
        'lga9PLwhGCs080xgcEBFxj3yVMPJAmmjRoNtAqMJxPVS68jZj9XhdY1m0gmuXY8Q9aYMC+gxvowcf/PlkktxfrxB6x94zlq8hm67'
        'wTfUFdMga1U21K3wFXlIVZG6XR6cUOQadEMeV013qGi0TewRkdfD9rnqSDMaCcGRMFxITw9DhW6Ia+55HcdOAosaXLHWgcMFYnaJ'
        'Bowu2IYzVRxJI7PPoQKB3V0vl90BKqlqctZKoLKIKyo+R1ltTzTcDU1AIY5B8QVmDa/YU39lSeE297jBrvCNBYAi+wQd/cgJBcPH'
        '3fP0bfPhsPu2Vgx0i+YHnnTl+b6GN9z9wNsS3BXNwVG+sLuG156nESBEcVH2FsNQQ1ulS2qcMQIMC2sMsIjdcfNh+onOMXwq6slk'
        'bDUiXtb4Vi6BagPIpwFXIL3V4Iq05EEbGdCmCxJI/RW4ob30UKWpEIPp3s3z4+ZuPlzG3SGn7iqkkUzQMQ8mM7frI63TPXXwQmGs'
        'T3ckYM76SGWD0by4GAbaVyVefn1i7X+kPC/iO+IJCOhIY8KncfuyskLxdgHzS17loFmv4a3/DU04uLS8ghIIyRpdkad5TtELRKrM'
        'gpYqBdzwphWUolaNtyN8XdDb2935+We00fQ7z80Hr/DYleh+C7yhbTm/4O2xC5WMNjW61o0jIhiTB5dttBW2TTOGKpbUsRUUDMax'
        'cT8ivczf9+scI2gcci5OIPHZvoa3CcZSLkJL4VS2MTToVir4J+JfprZJyKEGt8lF0yqGNmIqxx0U8Pn5NH/f3O/2e/yI1N/PrxhW'
        'DSUvxA81vgtxvKeBM3VGQ+BUSF2f+kKBmajFgjhDhabFkOQdQ3qubrHMOz7X0SIYzZvPwSCpiu73m7v9nDXCWFoLwj7bp8WOkLaG'
        'FrZOpMnN2+wK12ATW5g0xAKUioqQvkZWXHnlSNehk+Jol7PbXq6nafPuNL5OjYPRIJGoh9R6CFrjNOjC2JjYeXtKQIvXaInTwrOL'
        'Ha3Q0B+H7DXa3zTgwhuCphZTxhaTHUcKucyn8YkZv+TdccQrHimHOMyynbSsb/GVNPQy4ww5EGnaWcETdcgTXZORZc8jqLtu0YU7'
        'BiLB1UHn60EncofhbT+t1Iysge4q7/SF8glXuApBt+3iTBpfHjIqUXQqrkN1DCS+Cz0kWOFmOHRQIajOLC+FQN7Oc71OSjJHtreq'
        'yoRaFmQhSJbi3YiePN0JrSpkIolSiCJohyiviNMFV1jSuBjI7dmB5MpnnizvrieqN40jHW1HUMPiBoYzFJqQ1YHKo/wRABooxV67'
        'G3i+fUUitCo2QHwBlDZbdBVxkrtUHxc20cjQvR8PT4A/tKpFwkZK0qK6NDPU2MIZLRK6TNxJjmMkrBqZnawpL+GZaSkg0IBWwIop'
        'x6JZ1ivsZvx2vx8PeNzcEEXBgEODjUTZKJTeAq3EwN+FREqlHOeQRQXNfvUkG3RoPq2lBZJ7ARai6CK85iHHFqjr3owYHT/jGvaN'
        'Ty3tn2B/7lcFqm2FrbIAt7bxe0jyVWigObws1QfaidssXTvU0Eq8vN+jEhuSA/D3f+ansYqwh+N8upyptIJsQJdkkwDQ/GcsU12g'
        '0JWgxtO5TEBVSDBdgPR5IybxdKtIXhnIPBckhhPaitrykUugP/nn9fW4+TQfp5aqdbFFdvESol22Qtdkre01t9PKpUvAqQpb0Y2f'
        'J4YhNnN8Cwjjgq0Zk1oCf4tS+XZt6N5eX79+nVD+zMtTy5p2xFQWdE4MbmjhNW1uuZBvQ+4YMGq26EI8KKp7gmSc0xQm2QZec4/5'
        '0cjYKvINoqH6NJ1358t42JK784IxHbFUBfVQ9vkCUbU+0jie+m1ac6eRWqBHuDlQjFA8q6Pd1VnXCIb1gUY0vGjF1drcDZIk+Z/t'
        'EqHYjehFI5snQkGfd5sDzSVY+qAp6DNzzooUy+2BYoLnXRoJU+cJyw8rfG2B1hT26LdMwYvu1x+0e17JByQwWNCklGcH3HmCNqxj'
        'J4saVMz0qkALX8ujYBhir84qg38SsCYa+4OYH+MDcZfXw4En9Wm/yiVaUdGkzxY+P9e2+Jqw8VHsXGqXq8ebWnxhTSOOo+KSv+QL'
        'FLgGXVNXPMIaXVUBBO67eXxFUO/LN/TydENqL+kiDA26YT7whQ+qbGtEEC28EKfvMvwxVqdPkwL6rcE1b4iP0HX1CDTaw0xaO+73'
        '5xVxThlwi805IOgWX1MXnLikiuuOyNys4JVSuDpQs1b6GqSmBt7oRdLqCNWkRD+tasbjTVahlSlSRW5oaEUzHhuqOi6KfK+zI0IE'
        'NVI29OF6+ZJDb5S0kxmPrRiodNGGYulz/wSL9kifpSMAAA=='
    ),
    'suppliers_dup.csv': (
        'H4sIAOIO2WoC/ysuLSjIyUwtis9M0clLzE3VyUlNTIkvycxNjU9JrCzWyc+DcIoSS1K5gg0MdYKhGhQcdYx1DPQsgIJGCEEnHUMT'
        'kKglUNgYIewMVmtpChQ1QYi6QBWDzDBFCLvqGBqAVBsBhc0Qwm4Q1ZYmQGFzhLA7kmoLhLAH1GxjnA7BaiMABvaLew8BAAA='
    ),
    'control_total.txt': (
        'H4sIAOIO2WoC/ws0VjAyMDJVKEotS80rTVUoSC1SSExOzi/NK8nMS7dSMDQ3N7U017Ow5AqEqkzJLAZKJZco5BelpBYVWymYm1tw'
        'AQCK/a5lRwAAAA=='
    ),
}
for _name, _b64 in _FILES.items():
    pathlib.Path(_name).parent.mkdir(parents=True, exist_ok=True)
    pathlib.Path(_name).write_bytes(gzip.decompress(base64.b64decode(_b64)))
print('✓ הקבצים נוצרו:', ', '.join(_FILES))

<div dir="rtl">

---

# חלק א' - קוראים Traceback ובודקים שה-API קיים (8 דק')

## הרקע

כשיש **שגיאה**, פייתון מדפיס Traceback: רשימה של "מי קרא למי". הכלל מההרצאה: **קוראים מלמטה למעלה**.

- **השורה האחרונה** אומרת **מה קרה** (סוג השגיאה והסבר).
- השורה `File "...", line N` **הראשונה מלמטה ש"שייכת לכם"** אומרת **איפה בקוד שלכם** זה קרה.
- שורות שהנתיב שלהן בתוך ספרייה (`pandas` ושות') אפשר **לדלג עליהן**: הבאג כמעט אף פעם לא שם.

כאן יש חמש תוכניות קצרות שנכשלות (e1 עד e5 מהמעבדה בהרצאה, בלי קבצים: הקוד של כל אחת כתוב בתא). הפונקציה `capture` מריצה קוד ומחזירה את טקסט ה-Traceback:

</div>

In [ ]:
E_CODES = {
    "e1.py": """import pandas as pd
df = pd.read_csv("orders_il.csv")
print(df["Quantity"].sum())""",
    "e2.py": """import pandas as pd
df = pd.read_csv("orders_il.csv", dtype=str)
print(df["quantity"].sum() + 5)""",
    "e3.py": """import pandas_excel_fast""",
    "e4.py": """import pandas as pd
df = pd.read_csv("orders_il.csv")
df.to_excel_fast("out.xlsx")""",
    "e5.py": """import pandas as pd
df = pd.read_csv("data/orders.csv")""",
}

def capture(code, filename):
    """Runs `code` as if it were the file `filename`.
    Returns the Traceback text if it fails, or None if it runs fine."""
    linecache.cache[filename] = (len(code), None, code.splitlines(True), filename)
    try:
        exec(compile(code, filename, "exec"), {})
    except Exception as e:
        return "".join(traceback.format_exception(type(e), e, e.__traceback__.tb_next))
    return None

print("✓ חמש התוכניות והפונקציה capture הוגדרו")

<div dir="rtl">

**לפני שמריצים את e1:** הקוד מבקש `df["Quantity"]`. מה תהיה השורה האחרונה של ה-Traceback, ואיזו שורה (קובץ ומספר שורה) תהיה "השורה שלכם"? (הסתכלו בתא למעלה: באיזו שורה של e1.py כתוב `Quantity`?)

**התחזית שלי:** _(כתבו כאן)_

</div>

In [ ]:
tb1 = capture(E_CODES["e1.py"], "e1.py")
print(tb1)

<div dir="rtl">

## שלב 1 - `parse_traceback`: מהטקסט אל "סוג, קובץ, שורה"

ממשו `parse_traceback(text)` שמקבלת טקסט של Traceback ומחזירה שלישייה `(kind, path, lineno)`:

- `kind` - סוג השגיאה (`"KeyError"`), מהשורה **האחרונה**.
- `path`, `lineno` - הקובץ ומספר השורה של **ה-File הראשון מלמטה שאינו בספרייה**. אם אין כזה: `None, None`.

**כלים:**
- `lines = text.strip().splitlines()` פותח את הטקסט לרשימת שורות. `lines[-1]` היא השורה האחרונה.
- `"KeyError: 'Quantity'".split(":")[0]` נותן `"KeyError"` (מה שלפני הנקודתיים הראשונות).
- `for line in reversed(lines):` עובר על השורות **מלמטה למעלה**. שורת מיקום מתחילה (אחרי רווחים) ב-`File "`: בדקו עם `line.strip().startswith('File "')`.
- משורה כזו: `line.split('"')[1]` נותן את הנתיב, ו-`int(line.split("line ")[1].split(",")[0])` נותן את מספר השורה.
- הפונקציה `is_library_path(path)` (נתונה, בתא למטה) אומרת אם נתיב הוא של ספרייה - **דלגו עליו**.

</div>

In [ ]:
def is_library_path(path):
    """True if path belongs to a library (pandas, python itself...) and not to your own code."""
    return ("site-packages" in path or "dist-packages" in path
            or "/lib/python" in path or path.startswith("<frozen"))

def parse_traceback(text):
    """text: a Python Traceback. Returns (kind, path, lineno):
    kind = the error type from the LAST line; path/lineno = the first 'File "...", line N' line,
    reading from the bottom up, whose path is NOT a library (is_library_path). None, None if there is none.
    e.g. for 'Traceback (most recent call last):\n  File "a.py", line 7, in <module>\n    x = d["q"]\nKeyError: \'q\''
         -> ("KeyError", "a.py", 7)
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - חלק א', שלב 1
S_E1 = """Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/indexes/base.py", line 3812, in get_loc
    return self._engine.get_loc(casted_key)
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
KeyError: 'Quantity'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "e1.py", line 3, in <module>
    print(df["Quantity"].sum())
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/frame.py", line 4102, in __getitem__
    indexer = self.columns.get_loc(key)
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/indexes/base.py", line 3819, in get_loc
    raise KeyError(key) from err
KeyError: 'Quantity'"""

S_NESTED = """Traceback (most recent call last):
  File "report.py", line 30, in <module>
    main()
  File "report.py", line 22, in main
    df = load("orders.csv")
  File "report.py", line 12, in load
    return pd.read_csv(path)
  File "/opt/conda/lib/python3.10/site-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
FileNotFoundError: [Errno 2] No such file or directory: 'orders.csv'"""

S_IMPORT = """Traceback (most recent call last):
  File "app.py", line 1, in <module>
    import fastexcel_pro
  File "<frozen importlib._bootstrap>", line 1027, in _find_and_load
ModuleNotFoundError: No module named 'fastexcel_pro'"""

S_TYPE = """Traceback (most recent call last):
  File "e2.py", line 3, in <module>
    print(df["quantity"].sum() + 5)
TypeError: can only concatenate str (not "int") to str"""

S_NONE = """Traceback (most recent call last):
  File "/usr/lib/python3.10/json/decoder.py", line 355, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)"""

def check_a1():
    assert parse_traceback(S_E1) == ("KeyError", "e1.py", 3), "parse_traceback: ב-Traceback של e1 צריך לצאת ('KeyError', 'e1.py', 3). קראו מלמטה והחליפו - דלגו על שורות הספרייה"
    assert parse_traceback(S_TYPE) == ("TypeError", "e2.py", 3), "parse_traceback: ב-e2 הסוג הוא TypeError (מה שלפני הנקודתיים הראשונות בשורה האחרונה)"
    assert parse_traceback(S_NESTED) == ("FileNotFoundError", "report.py", 12), "parse_traceback: כשיש כמה שורות שלכם - לוקחים את הראשונה מלמטה (הקרובה ביותר למקום הכשל), כאן שורה 12"
    assert parse_traceback(S_IMPORT) == ("ModuleNotFoundError", "app.py", 1), "parse_traceback: שורת <frozen ...> היא של פייתון עצמו - דלגו עליה"
    assert parse_traceback(S_NONE) == ("json.decoder.JSONDecodeError", None, None), "parse_traceback: אם כל שורות ה-File הן של ספרייה - מחזירים None, None"
    return True

check_a1() and print("✓ שלב 1: parse_traceback קוראת Traceback מלמטה למעלה")

In [ ]:
# מריצים את parse_traceback על חמש התוכניות
for name, code in E_CODES.items():
    tb = capture(code, name)
    if tb is None:
        print(f"{name}: לא נכשלה בסביבה הזו")
        continue
    kind, path, lineno = parse_traceback(tb)
    print(f"{name}:  {kind:<20} ← {path}, שורה {lineno}")

<div dir="rtl">

**(א)** למה קוראים Traceback **מלמטה** ולא מלמעלה? ובאיזה מקרה ב-e1 יש גם בלוק Traceback שני (השרשרת "The above exception was the direct cause")? איזה בלוק מעניין אתכם ולמה?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

## שלב 2 - האם זה בכלל קיים? (בדיקת API)

שתי מתוך חמש השגיאות (e3, e4) הן **הזיות מהטקסונומיה של מפגש 2**: ה-AI כתב שם של חבילה (`pandas_excel_fast`) ושם של מתודה (`to_excel_fast`) שנשמעים סבירים **ואינם קיימים**. הצעד הראשון בשגיאה כזו: **לבדוק אם זה קיים** - לא לבקש מה-AI "לנסות שוב".

ממשו שתי פונקציות קצרות:

- `api_exists(obj, name)` - האם לאובייקט (או למחלקה) יש תכונה או מתודה בשם `name`? **רמז:** `hasattr(obj, name)`.
- `package_exists(name)` - האם חבילה בשם הזה **מותקנת** בסביבה? **רמז:** `importlib.util.find_spec(name)` מחזירה `None` כשאין חבילה, ואחרת אובייקט.

> שימו לב: `package_exists` בודקת רק מה שמותקן אצלכם. כדי לדעת אם חבילה בכלל **קיימת בעולם** מחפשים את השם ב-pypi.org (אין עמוד: אין חבילה). וגם אם היא קיימת - התקנה של חבילה לא מוכרת מריצה קוד זר (מפגש 10).

</div>

In [ ]:
def api_exists(obj, name):
    """True if `obj` has an attribute/method called `name`.
    e.g. api_exists(pd.DataFrame, "to_csv") -> True ;  api_exists(pd.DataFrame, "to_csv_fast") -> False
    """
    # TODO
    raise NotImplementedError


def package_exists(name):
    """True if a package called `name` is installed here (can be imported).
    e.g. package_exists("pandas") -> True ;  package_exists("pandas_excel_fast") -> False
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - חלק א', שלב 2
def check_a2():
    assert api_exists(pd.DataFrame, "to_excel") is True, "api_exists: ל-DataFrame יש מתודה to_excel"
    assert api_exists(pd.DataFrame, "to_excel_fast") is False, "api_exists: to_excel_fast לא קיימת - החזירו False"
    assert api_exists(pd, "read_csv") is True, "api_exists: ל-pd יש read_csv (הפונקציה עובדת גם על מודול)"
    assert api_exists(pd, "read_csv_turbo") is False, "api_exists: read_csv_turbo לא קיימת"
    assert package_exists("pandas") is True, "package_exists: pandas מותקנת"
    assert package_exists("json") is True, "package_exists: json קיימת בפייתון"
    assert package_exists("pandas_excel_fast") is False, "package_exists: pandas_excel_fast לא קיימת - החזירו False (find_spec מחזירה None)"
    return True

check_a2() and print("✓ שלב 2: בדיקות קיום עובדות")

<div dir="rtl">

**לפני שמריצים:** מהשמות הבאים של מתודות ב-`pd.DataFrame`, אילו קיימים באמת? `to_excel`, `to_excel_fast`, `to_parquet`, `to_sql_fast`, `to_markdown`.

**התחזית שלי:** _(כתבו כאן)_

</div>

In [ ]:
for name in ["to_excel", "to_excel_fast", "to_parquet", "to_sql_fast", "to_markdown"]:
    print(f"pd.DataFrame.{name:<15} ", "קיים" if api_exists(pd.DataFrame, name) else "מומצא!")
print()
for name in ["openpyxl", "pandas_excel_fast"]:
    print(f"חבילה {name:<20} ", "מותקנת" if package_exists(name) else "לא נמצאה (בודקים ב-pypi.org)")

<div dir="rtl">

**(ב)** מה משותף ל-e3 ול-e4, לאיזה סוג הזיה מהטקסונומיה של מפגש 2 כל אחת שייכת, ולמה **לא** נבקש מה-AI "תנסה שוב" אלא נחפש בתיעוד את השם הנכון?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

---

# חלק ב' - משחזרים ומבודדים באג שקט (10 דק')

## הרקע

עכשיו הגרוע יותר: **אין שגיאה**. הדוח רץ, מדפיס מספר, והמספר שגוי בכ-50%. אין Traceback להיאחז בו, ולכן נשתמש בצעדים הראשונים של השיטה: **משחזרים** (מקבלים שוב את אותו מספר שגוי, ומשווים לסכום חיצוני), **מבודדים** (מקטינים את הבעיה) ו**בודקים השערות עם ספירות**, לא בקריאת הקוד.

הנתונים: `orders_il.csv` (3,040 שורות הזמנה, תאריכים בפורמט **יום/חודש/שנה**), `products.csv`, `suppliers_dup.csv`. סכום הביקורת של הנהלת החשבונות נמצא ב-`control_total.txt`:

</div>

In [ ]:
orders = pd.read_csv("orders_il.csv")
products = pd.read_csv("products.csv")
suppliers = pd.read_csv("suppliers_dup.csv")

print(open("control_total.txt").read())
CONTROL_TOTAL = 177597.89     # לפי הנהלת החשבונות
CONTROL_ORDERS = 778          # הזמנות ייחודיות ברבעון 3
print(len(orders), "שורות הזמנה,", len(products), "מוצרים,", len(suppliers), "שורות ספקים")
orders.head()

<div dir="rtl">

## שלב 1 - משחזרים

הפונקציה `buggy_report` היא **הקוד של הדוח שה-AI כתב** (אותה לוגיקה בדיוק כמו `report_buggy.py` מהחבילה, ארוזה בפונקציה). קראו אותה, אבל **אל תנסו לגלות את הבאגים בקריאה** - נגלה אותם במדידה.

</div>

In [ ]:
def buggy_report(orders, products, suppliers):
    """The AI-written report logic. Returns (q3, df, total)."""
    orders = orders.copy()
    orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
    orders["revenue"] = orders["quantity"] * orders["unit_price"]
    q3 = orders[(orders["order_date"] > "2025-07-01") & (orders["order_date"] <= "2025-09-30")]
    df = q3.merge(products[["product_id", "supplier_id"]], on="product_id")
    df = df.merge(suppliers[["supplier_id", "name"]], on="supplier_id")
    return q3, df, round(df["revenue"].sum(), 2)

print("✓ buggy_report הוגדרה")

<div dir="rtl">

**לפני ההרצה:** האם יהיה Traceback? מה יהיה הסכום ביחס ל-177,597.89 (קרוב, חצי, כפול)? וכמה הזמנות בערך ייכנסו לרבעון 3 - לפי הסכום הגיוני ש-778, או משהו אחר?

**התחזית שלי:** _(כתבו כאן)_

</div>

In [ ]:
q3_buggy, df_buggy, total_buggy = buggy_report(orders, products, suppliers)
print("סכום הדוח:    ", total_buggy)
print("סכום הביקורת: ", CONTROL_TOTAL)
print("הפער:         ", round(CONTROL_TOTAL - total_buggy, 2))
print("שורות ברבעון 3 (לפני צירוף):", len(q3_buggy), " | אחרי צירוף לספקים:", len(df_buggy))

<div dir="rtl">

**שחזרנו**: 88,526.81 לעומת 177,597.89, בלי שום הודעה. קראו עוד דבר: רק 278 שורות נכנסו לרבעון, אבל אחרי הצירוף יש 346. **מספר שורות שעולה בצירוף הוא סימן אזהרה** (מפגש 7).

## שלב 2 - בדיקות שפיות: ספירות במקום ניחושים

לפי ההרצאה, ארבע בדיקות שפיות: סכום ביקורת, **ספירות**, **ייחודיות**, טווחים. בלי להבין עדיין את הקוד של הדוח, נכתוב שתי פונקציות קטנות שמודדות:

- `count_nat(orders)` - כמה תאריכים `pd.to_datetime(..., errors="coerce")` **לא הצליח לקרוא** (`NaT` = "לא תאריך").
  **רמז:** `pd.to_datetime(orders["order_date"], errors="coerce")` נותן עמודה; `.isna()` נותן True/False לכל שורה; `.sum()` סופר את ה-True.
- `count_dup_ids(df, col)` - כמה שורות בעמודה `col` הן **חזרה** על ערך שכבר הופיע.
  **רמז:** `df[col].duplicated()` מסמן True לכל הופעה שנייה ואילך. 

</div>

In [ ]:
def count_nat(orders):
    """Returns how many values of orders["order_date"] become NaT with the default reading:
    pd.to_datetime(orders["order_date"], errors="coerce")
    """
    # TODO
    raise NotImplementedError


def count_dup_ids(df, col):
    """Returns how many rows of df[col] repeat a value that already appeared above.
    e.g. the values 1, 2, 2, 3, 3, 3 -> 3
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - חלק ב', שלב 2
def check_b1():
    t = pd.DataFrame({"order_date": ["03/04/2025", "23/01/2025", "31/12/2025"]})
    assert count_nat(t) == 2, "count_nat: בדוגמה הקטנה שני התאריכים 23/01 ו-31/12 לא נקראים (אין חודש 23 או 31), ולכן 2"
    assert count_dup_ids(pd.DataFrame({"k": [1, 2, 2, 3, 3, 3]}), "k") == 3, "count_dup_ids: 1,2,2,3,3,3 - שלוש חזרות (השני של 2, השני והשלישי של 3)"
    assert count_dup_ids(pd.DataFrame({"k": ["a", "b"]}), "k") == 0, "count_dup_ids: אם הכול ייחודי צריך להחזיר 0"
    assert count_nat(orders) == 1847, "count_nat: על orders_il.csv צריכים לצאת 1,847 תאריכים שלא נקראו"
    return True

check_b1() and print("✓ שלב 2: שתי בדיקות השפיות עובדות")

<div dir="rtl">

**לפני שמריצים על הנתונים האמיתיים (3,040 שורות):** כמה תאריכים יהיו NaT? כמה `order_id` חוזרים? כמה שורות חוזרות בטבלת הספקים (10 שורות)? (רמז: תאריך שמתחיל ב-23 או 31 - מה יקרה אם פייתון מנסה לקרוא אותו כ-חודש/יום?)

**התחזית שלי:** _(כתבו כאן)_

</div>

In [ ]:
print("תאריכים שלא נקראו (NaT):      ", count_nat(orders), "מתוך", len(orders))
print("order_id חוזרים:               ", count_dup_ids(orders, "order_id"))
print("supplier_id חוזרים בספקים:     ", count_dup_ids(suppliers, "supplier_id"))

<div dir="rtl">

## שלב 3 - מבודדים: איפה בדיוק השורות הבעייתיות?

מספר כמו 1,847 עוד לא מסביר **למה**. ממשו `find_bad_rows(orders)` שמחזירה **את השורות עצמן** (טבלה) שהתאריך שלהן הופך ל-`NaT`, כדי שנוכל להסתכל עליהן ולמצוא מה משותף להן.

**רמז:** סינון בעזרת מסכה: אם `mask` היא עמודת True/False באותו אורך, אז `orders[mask]` מחזיר רק את השורות שבהן True. את המסכה כבר בניתם בשלב הקודם (`.isna()`).

</div>

In [ ]:
def find_bad_rows(orders):
    """Returns the rows (a DataFrame, all original columns) of `orders` whose order_date
    becomes NaT with pd.to_datetime(..., errors="coerce"). Does not change `orders`.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - חלק ב', שלב 3
def check_b2():
    t = pd.DataFrame({"order_id": ["A", "B", "C"], "order_date": ["03/04/2025", "23/01/2025", "31/12/2025"]})
    r = find_bad_rows(t)
    assert isinstance(r, pd.DataFrame), "find_bad_rows: החזירו טבלה (DataFrame) של השורות, לא מספר"
    assert list(r["order_id"]) == ["B", "C"], "find_bad_rows: בדוגמה הקטנה השורות הבעייתיות הן B ו-C"
    assert len(t) == 3, "find_bad_rows: אסור לשנות את הטבלה המקורית"
    bad = find_bad_rows(orders)
    assert len(bad) == 1847, "find_bad_rows: על orders_il.csv צריכות לצאת 1,847 שורות"
    assert list(bad.columns) == list(orders.columns), "find_bad_rows: החזירו את כל העמודות המקוריות"
    return True

check_b2() and print("✓ שלב 3: מצאנו את", len(find_bad_rows(orders)), "השורות הבעייתיות")

In [ ]:
bad = find_bad_rows(orders)
print(bad.head())
first_part = bad["order_date"].str[:2].astype(int)     # המספר הראשון בכל תאריך
print("\nהמספר הראשון בתאריך, בכל השורות הבעייתיות: הקטן ביותר =", first_part.min(), "| הגדול ביותר =", first_part.max())

<div dir="rtl">

**(א)** מה משותף לכל 1,847 השורות? מה זה אומר על הפורמט שפייתון **ניחש** (חודש/יום או יום/חודש)? ומה קורה לשורה עם `NaT` בסינון `(order_date > ...) & (order_date <= ...)` - האם יש שגיאה?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

## שלב 4 - מקרה מינימלי

הצעד "מבודדים" מסתיים במקרה **קטן** שמראה את הבאג, כזה שאפשר להבין ולהדביק ל-AI. בטבלה הגדולה יש **שני** כשלים שונים בתאריכים:

1. **`NaT`**: יום גדול מ-12 (למשל `23/01/2025`) - התאריך נעלם.
2. **היפוך שקט**: יום קטן מ-13 (למשל `03/04/2025`, שהוא 3 באפריל) נקרא כ-**4 במרץ**, בלי שום סימן.

בחרו `mini` - טבלה של **לא יותר מ-5 שורות** מתוך `orders` (עם העמודות `order_id` ו-`order_date`) שמראה **את שני הכשלים**. **רמז:** `orders.head(5)` או `orders.iloc[[...]]`.

> **שימו לב:** pandas לומד את הפורמט מהשורה **הראשונה** בעמודה. אם השורה הראשונה שלכם היא עם יום גדול מ-12 (כמו `23/01/2025`), הוא יניח יום/חודש וידפיס הכול נכון - ואז לא תראו את הבאג. התא שאחרי הבחירה יראה לכם את התוצאה.

</div>

In [ ]:
# בחרו עד 5 שורות שמראות את שני הכשלים
mini = orders[["order_id", "order_date"]].head(0)   # TODO: החליפו ב-head(5) או בבחירה משלכם

In [ ]:
show = mini.copy()
show["default_reading"] = pd.to_datetime(mini["order_date"], errors="coerce")    # מה שהדוח עשה
show["real_date_dd_mm"] = pd.to_datetime(mini["order_date"], format="%d/%m/%Y")   # מה שהתאריך באמת
show

In [ ]:
# בדיקה - חלק ב', שלב 4
def check_b3():
    assert 1 <= len(mini) <= 5, "mini: מקרה מינימלי הוא עד 5 שורות"
    default = pd.to_datetime(mini["order_date"], errors="coerce")
    real = pd.to_datetime(mini["order_date"], format="%d/%m/%Y")
    assert default.isna().sum() >= 1, "mini: צריכה להיות לפחות שורה אחת שהופכת ל-NaT (יום גדול מ-12). זכרו: השורה הראשונה קובעת את הפורמט שפנדס מנחש"
    assert ((default != real) & default.notna()).sum() >= 1, "mini: צריכה להיות לפחות שורה אחת שנקראת הפוך בשקט (יום קטן מ-13 ושונה מהחודש)"
    return True

check_b3() and print("✓ שלב 4: מקרה מינימלי בגודל", len(mini), "שורות מראה את שני הכשלים")

<div dir="rtl">

**(ב)** סכום הביקורת (177,597.89) אמר לנו **שיש** בעיה. מה נתנו לנו הספירות (NaT, order_id חוזרים, ספקים חוזרים) שסכום הביקורת לא נתן? ולמה מקרה מינימלי של 5 שורות עדיף, גם לנו וגם אם נדביק את הבעיה ל-AI?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

---

# חלק ג' - מתקנים, בדיקת רגרסיה, תחקיר (10 דק')

## הרקע

יש ארבע סיבות לפער (שלוש ראינו כבר בספירות; את הרביעית נראה בשלב 1): **(1)** התאריכים נקראו בלי פורמט, **(2)** הטווח `>` מוציא את 1 ביולי, **(3)** הזמנות כפולות, **(4)** ספקים כפולים. כלל השיעור: **לא מתקנים סימפטום.** כל סיבה מקבלת תיקון **קטן אחד**, ואחרי כל תיקון **מאמתים** מול סכום הביקורת.

## שלב 1 - שני תיקונים: תאריך וטווח

ממשו:

- `parse_dates(orders)` - מחזירה **עותק** של הטבלה שבו `order_date` נקרא בפורמט **יום/חודש/שנה**. **רמז:** `pd.to_datetime(עמודה, format="%d/%m/%Y")`. **בלי** `errors="coerce"`: תאריך לא תקין צריך להפיל את התוכנית בקול (שגיאה) ולא להיעלם בשקט.
- `in_q3(df)` - מקבלת טבלה שהתאריכים בה כבר נקראו, ומחזירה רק את השורות מ-**1 ביולי 2025 עד 30 בספטמבר 2025, כולל שני הקצוות**. **רמז:** שני תנאים עם `&`, וסימן ההשוואה שכולל קצה הוא `>=` (ו-`<=`).

</div>

In [ ]:
def parse_dates(orders):
    """Returns a COPY of orders where order_date is read as day/month/year (format="%d/%m/%Y").
    No errors="coerce": an invalid date must raise an error.
    """
    # TODO
    raise NotImplementedError


def in_q3(df):
    """df: order_date is already datetime. Returns the rows from 2025-07-01 up to 2025-09-30, BOTH ends included."""
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - חלק ג', שלב 1
def check_c1():
    t = pd.DataFrame({"order_date": ["03/04/2025", "23/01/2025"]})
    r = parse_dates(t)
    assert list(r["order_date"]) == [pd.Timestamp("2025-04-03"), pd.Timestamp("2025-01-23")], "parse_dates: 03/04/2025 הוא 3 באפריל (יום/חודש), ו-23/01/2025 הוא 23 בינואר"
    assert t["order_date"].iloc[0] == "03/04/2025", "parse_dates: אסור לשנות את הטבלה המקורית - החזירו עותק"
    try:
        parse_dates(pd.DataFrame({"order_date": ["לא תאריך"]}))
    except ValueError:
        pass
    else:
        raise AssertionError("parse_dates: תאריך לא תקין צריך להפיל שגיאה. אל תשתמשו ב-errors=\"coerce\" - הוא מסתיר את הכשל")
    d = pd.DataFrame({"order_date": pd.to_datetime(["2025-06-30", "2025-07-01", "2025-09-30", "2025-10-01"])})
    assert len(in_q3(d)) == 2, "in_q3: צריכות להישאר שתי שורות - 1 ביולי ו-30 בספטמבר (שני הקצוות כלולים)"
    assert pd.Timestamp("2025-07-01") in list(in_q3(d)["order_date"]), "in_q3: 1 ביולי חייב להיכלל (>= ולא >)"
    return True

check_c1() and print("✓ שלב 1: parse_dates ו-in_q3 עובדות")

<div dir="rtl">

**לפני שמריצים:** כשהתאריכים נקראים נכון, כמה הזמנות נרשמו ב-1 ביולי 2025? ומה יקרה לכולן בסינון `order_date > "2025-07-01"` של הדוח הפגום?

**התחזית שלי:** _(כתבו כאן)_

</div>

In [ ]:
fixed_dates = parse_dates(orders)
print("הזמנות ב-1 ביולי 2025:", int((fixed_dates["order_date"] == "2025-07-01").sum()))

<div dir="rtl">

## שלב 2 - שני תיקונים: כפילויות

ממשו שתי פונקציות. בשתיהן **הרמז:** `df.drop_duplicates("עמודה")` משאיר רק את ההופעה הראשונה של כל ערך בעמודה (ומחזיר טבלה חדשה).

- `drop_dup_orders(orders)` - שורה אחת לכל `order_id`.
- `drop_dup_suppliers(suppliers)` - שורה אחת לכל `supplier_id` (אחרת הצירוף **מכפיל** הזמנות).

</div>

In [ ]:
def drop_dup_orders(orders):
    """Returns the orders with one row per order_id (the first one is kept)."""
    # TODO
    raise NotImplementedError


def drop_dup_suppliers(suppliers):
    """Returns the suppliers with one row per supplier_id (the first one is kept)."""
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - חלק ג', שלב 2
def check_c2():
    o = pd.DataFrame({"order_id": ["A", "B", "A"], "quantity": [1, 2, 1]})
    r = drop_dup_orders(o)
    assert list(r["order_id"]) == ["A", "B"], "drop_dup_orders: צריך להישאר A ו-B, פעם אחת כל אחד"
    assert len(o) == 3, "drop_dup_orders: אסור לשנות את הטבלה המקורית"
    s = pd.DataFrame({"supplier_id": ["S1", "S2", "S2"], "name": ["x", "y", "y"]})
    assert list(drop_dup_suppliers(s)["supplier_id"]) == ["S1", "S2"], "drop_dup_suppliers: צריך להישאר S1 ו-S2, פעם אחת כל אחד"
    assert len(drop_dup_orders(orders)) == 3000 and len(drop_dup_suppliers(suppliers)) == 8, "ב-orders_il.csv יש 3,000 הזמנות ייחודיות, ובספקים 8"
    return True

check_c2() and print("✓ שלב 2: שתי פונקציות הכפילויות עובדות")

<div dir="rtl">

## שלב 3 - פנקס תיקונים: מאמתים אחרי כל תיקון

הפונקציה `q3_revenue` (נתונה) היא הדוח המתוקן: לכל תיקון יש "מתג" ברשימה `fixes`. אם התיקון **לא** ברשימה, הדוח מתנהג כמו הגרסה הפגומה. כך נוכל להפעיל את התיקונים **אחד אחד** ולמדוד את הסכום אחרי כל אחד - צעד 6 בשיטה, **מאמתים**.

</div>

In [ ]:
ALL_FIXES = ("dates", "range", "orders", "suppliers")

def q3_revenue(orders, products, suppliers, fixes=()):
    """The report. `fixes` is a tuple with any of: "dates", "range", "orders", "suppliers".
    Returns the joined table (one row per order line), with a revenue column."""
    orders = orders.copy()
    if "orders" in fixes:
        orders = drop_dup_orders(orders)
    if "suppliers" in fixes:
        suppliers = drop_dup_suppliers(suppliers)
    if "dates" in fixes:
        orders = parse_dates(orders)
    else:
        orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
    orders["revenue"] = orders["quantity"] * orders["unit_price"]
    if "range" in fixes:
        q3 = in_q3(orders)
    else:
        q3 = orders[(orders["order_date"] > "2025-07-01") & (orders["order_date"] <= "2025-09-30")]
    df = q3.merge(products[["product_id", "supplier_id"]], on="product_id")
    return df.merge(suppliers[["supplier_id", "name"]], on="supplier_id")

print("✓ q3_revenue הוגדרה")

<div dir="rtl">

**לפני שמריצים את הפנקס:** אחרי התיקון **הראשון** (תאריכים בלבד), הסכום יהיה מתחת ל-177,597.89, מעליו, או שווה? ולמה? (רמז: מה עדיין לא תוקן?)

**התחזית שלי:** _(כתבו כאן)_

</div>

In [ ]:
steps = [("0. before fixes", ()),
         ("1. dates dd/mm/yyyy", ("dates",)),
         ("2. range includes Jul 1", ("dates", "range")),
         ("3. drop duplicate orders", ("dates", "range", "orders")),
         ("4. drop duplicate suppliers", ALL_FIXES)]
print(f"{'after the fix':<30}{'total':>12}{'rows':>7}{'vs control':>13}")
for label, fx in steps:
    df = q3_revenue(orders, products, suppliers, fx)
    total = round(df["revenue"].sum(), 2)
    print(f"{label:<30}{total:>12,.2f}{len(df):>7}{total - CONTROL_TOTAL:>+13,.2f}")

In [ ]:
# בדיקה - חלק ג', שלב 3
def check_c3():
    expected = [88526.81, 221849.03, 225051.21, 222175.27, 177597.89]
    for (label, fx), exp in zip(steps, expected):
        got = round(q3_revenue(orders, products, suppliers, fx)["revenue"].sum(), 2)
        assert got == exp, f"הפנקס: אחרי ״{label}״ ציפינו ל-{exp:,.2f} והתקבל {got:,.2f} - בדקו את הפונקציה של התיקון הזה"
    final = q3_revenue(orders, products, suppliers, ALL_FIXES)
    assert len(final) == CONTROL_ORDERS, f"הדוח המתוקן: צריכות להיות {CONTROL_ORDERS} שורות (שורה לכל הזמנה)"
    return True

check_c3() and print("✓ שלב 3: הפנקס תואם - הסכום הסופי הוא סכום הביקורת")

<div dir="rtl">

## שלב 4 - למה לא סתם "לכוון" את המספר?

אפשר היה לחשוב על קיצור: הפער הוא 89,071.08, אז נוסיף אותו לסכום הפגום ונסיים. בואו נראה:

</div>

In [ ]:
gap = round(CONTROL_TOTAL - total_buggy, 2)
print("הסכום הפגום + הפער =", round(total_buggy + gap, 2), "  ← שווה לסכום הביקורת?", round(total_buggy + gap, 2) == CONTROL_TOTAL)

<div dir="rtl">

**לפני הצעד הבא:** ניקח רק את **מחצית** ההזמנות (1,500 הראשונות). הדוח המתוקן נותן סכום נכון למחצית הזו. האם הטלאי 'להוסיף 89,071.08' לדוח הפגום ייתן את אותו סכום?

**התחזית שלי:** _(כתבו כאן)_

</div>

In [ ]:
half = orders.iloc[:1500]
fixed_half = round(q3_revenue(half, products, suppliers, ALL_FIXES)["revenue"].sum(), 2)
patched_half = round(q3_revenue(half, products, suppliers)["revenue"].sum() + gap, 2)
print("הדוח המתוקן על חצי מהנתונים:       ", fixed_half)
print("הדוח הפגום + 89,071.08 על אותם נתונים:", patched_half)

<div dir="rtl">

**(א)** הטלאי "להוסיף את הפער" נתן סכום נכון על כל הנתונים. למה הוא **לא** תיקון? (חשבו על: סימפטום מול שורש, ומה קורה כשמגיעים נתונים חדשים.) ואיזה סימן בקוד של AI היה מזהיר אתכם שהוא מטלא?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

## שלב 5 - בדיקת רגרסיה: "תיקון בלי טסט רגרסיה הוא תיקון זמני"

תיקנו, והסכום מתאים. אבל בעוד חודש מישהו (או ה-AI) יערוך את הדוח, והבאג עלול לחזור **בשקט**. **בדיקת רגרסיה** היא קוד שנכשל כשהבאג חוזר. בדיקה טובה:

1. **נכשלת על הקוד הישן** ועוברת על המתוקן (אחרת היא לא מוכיחה כלום).
2. בודקת מול **מקור חיצוני** (הנהלת החשבונות) - לא מול מספר שנגזר מהתוצאה של הקוד עצמו.

ממשו `test_regression(get_df)`. הפרמטר `get_df` הוא **פונקציה ללא ארגומנטים** שמחזירה את טבלת הדוח (כך הבדיקה יכולה לרוץ על גרסות שונות של הדוח). שלוש בדיקות עם `assert תנאי, "הודעה"`:

1. סכום ההכנסה (`df["revenue"].sum()`), מעוגל ל-2 ספרות, שווה ל-`CONTROL_TOTAL`.
2. `order_id` ייחודי (`df["order_id"].is_unique`).
3. יש לפחות הזמנה אחת ב-1 ביולי: `(df["order_date"] == "2025-07-01").any()`.

(בפרויקט אמיתי זו פונקציה בקובץ `test_report.py` ש-`pytest` מריץ אוטומטית - מפגש 8. כאן נריץ אותה ידנית.)

</div>

In [ ]:
def test_regression(get_df):
    """get_df() returns the report table. Raise AssertionError (with a message) if:
    (1) round(total revenue, 2) != CONTROL_TOTAL, (2) order_id is not unique, (3) there is no order on 2025-07-01.
    Return True if all three pass.
    """
    df = get_df()
    # TODO: three assert lines
    raise NotImplementedError

In [ ]:
# בדיקה - חלק ג', שלב 5 (הבדיקה בודקת את הבדיקה שלכם)
def run_with(fixes):
    return lambda: q3_revenue(orders, products, suppliers, fixes)

def check_c4():
    assert test_regression(run_with(ALL_FIXES)) is True, "test_regression: על הדוח המתוקן הבדיקה צריכה לעבור ולהחזיר True"
    versions = [("הדוח הפגום (בלי תיקונים)", ())]
    for fix in ALL_FIXES:
        versions.append((f"הדוח בלי התיקון ״{fix}״", tuple(f for f in ALL_FIXES if f != fix)))
    for label, fx in versions:
        try:
            test_regression(run_with(fx))
        except AssertionError:
            continue
        raise AssertionError(f"test_regression: על ״{label}״ הבדיקה עברה - היא חלשה מדי. בדיקה טובה נכשלת כשהבאג חוזר")
    return True

check_c4() and print("✓ שלב 5: הטסט נכשל על 5 גרסאות פגומות ועובר על המתוקנת")

In [ ]:
# מה הבדיקה שלכם אומרת על כל גרסה?
versions = [("buggy (no fixes)", ())]
for fix in ALL_FIXES:
    versions.append((f"without '{fix}' fix", tuple(f for f in ALL_FIXES if f != fix)))
versions.append(("all four fixes", ALL_FIXES))

for label, fx in versions:
    try:
        test_regression(run_with(fx))
        result = "עברה"
    except AssertionError as e:
        result = f"נכשלה: {e}"
    print(f"{label:<28} {result}")

<div dir="rtl">

**(ב)** למה הכלל אומר שהבדיקה חייבת **להיכשל** על הקוד הישן? מה היה קורה אם הבדיקה הייתה `assert df["revenue"].sum() > 100000`? ומה היתרון של הבדיקה על פני תיעוד של "הסכום צריך להיות 177,597.89"?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

## שלב 6 - תחקיר ו-commit (אפשר להשלים בבית)

סוגרים את המעגל בשני טקסטים קצרים:

**תחקיר באג** (עמוד אחד בפרויקט הקבוצתי; כאן חמש שורות). שימו לב: **שורש הבעיה הוא סיבה, לא סימפטום.** "הסכום נמוך" זה סימפטום; "הקוד קרא תאריכים בלי לציין פורמט" זה שורש.

**הודעת commit** שמסבירה **למה** ולא רק מה. בשורה הראשונה (עד 72 תווים) מה שונה, ובשורות אחריה למה. כתבו באנגלית, והזכירו בה את המילים `date` ו-`duplicate`. דוגמה לצורה: `Fix X: do A and B` ואחריה שורה `Why: ...`.

</div>

In [ ]:
postmortem = {
    "what":       "(מה קרה, בשורה אחת)",
    "found_by":   "(איך גילינו)",
    "root_cause": "(השורש, לא הסימפטום)",
    "fix":        "(מה שונה ולמה)",
    "prevention": "(בדיקה או כלל חדש)",
}

commit_message = """(כתבו כאן הודעת commit: שורה ראשונה קצרה, ואחריה שורה שמסבירה למה)"""

In [ ]:
# בדיקה - חלק ג', שלב 6
def check_c5():
    assert set(postmortem) == {"what", "found_by", "root_cause", "fix", "prevention"}, "postmortem: השאירו את חמשת המפתחות"
    for key, value in postmortem.items():
        assert isinstance(value, str) and len(value.strip()) >= 12 and not value.strip().startswith("("), f"postmortem: מלאו את ״{key}״ (משפט קצר, ולא את הטקסט שבסוגריים)"
    lines = [l for l in commit_message.splitlines() if l.strip()]
    assert len(lines) >= 2 and not lines[0].startswith("("), "commit_message: שורה ראשונה קצרה ושורה שמסבירה למה"
    assert len(lines[0]) <= 72, "commit_message: השורה הראשונה עד 72 תווים"
    low = commit_message.lower()
    assert "date" in low and ("duplicate" in low or "dedup" in low), "commit_message: הזכירו date ו-duplicate (באנגלית) - כך ברור מה תוקן"
    return True

check_c5() and print("✓ שלב 6: תחקיר והודעת commit מוכנים")

<div dir="rtl">

---

# חלק ד' - טקסונומיה של באגים (3 דק')

## הרקע

בהרצאה חילקנו באגים בקוד שנוצר ב-AI לשתי משפחות:

- `"hallucination"` - **הזיה**: ה-AI המציא API שלא קיים (פונקציה, מתודה, פרמטר, חבילה).
- `"assumption"` - **הנחה שגויה** על הנתונים או על הסביבה: שם עמודה, סוג נתון, נתיב קובץ, פורמט תאריך, כפילויות, גבולות טווח.

ובנוסף שאלה אחת שחשובה לא פחות: האם הבאג **רועש** (יש שגיאה ותוכנית שנופלת) או **שקט** (עובר לדוח)?

תייגו את **תשעת** הבאגים שפגשנו: חמש השגיאות מחלק א' ואת ארבעת הבאגים של הדוח. החליפו את `"?"` ב-`"hallucination"` / `"assumption"` ובמילון השני ב-`True` / `False` (האם שקט).

| מזהה | מה זה |
|---|---|
| `e1` | KeyError: `Quantity` מול `quantity` |
| `e2` | TypeError: העמודה נקראה כטקסט (`dtype=str`) |
| `e3` | ModuleNotFoundError: `pandas_excel_fast` |
| `e4` | AttributeError: `to_excel_fast` |
| `e5` | FileNotFoundError: `data/orders.csv` |
| `r1_dates` | תאריכים נקראו יום/חודש הפוך, עם `coerce` |
| `r2_range` | `>` במקום `>=` ביום הראשון |
| `r3_dup_orders` | הזמנות כפולות |
| `r4_dup_suppliers` | ספקים כפולים בצירוף |

</div>

In [ ]:
family = {
    "e1": "?", "e2": "?", "e3": "?", "e4": "?", "e5": "?",
    "r1_dates": "?", "r2_range": "?", "r3_dup_orders": "?", "r4_dup_suppliers": "?",
}

is_silent = {
    "e1": "?", "e2": "?", "e3": "?", "e4": "?", "e5": "?",
    "r1_dates": "?", "r2_range": "?", "r3_dup_orders": "?", "r4_dup_suppliers": "?",
}

In [ ]:
# בדיקה - חלק ד'
def check_d():
    fam = {"e1": "assumption", "e2": "assumption", "e3": "hallucination", "e4": "hallucination", "e5": "assumption",
           "r1_dates": "assumption", "r2_range": "assumption", "r3_dup_orders": "assumption", "r4_dup_suppliers": "assumption"}
    why = {"e1": "שם עמודה שגוי הוא הנחה על הנתונים", "e2": "הנחה על סוג הנתון (טקסט במקום מספר)",
           "e3": "חבילה שלא קיימת ב-PyPI - הזיית חבילה", "e4": "מתודה שלא קיימת ב-pandas - הזיית מתודה",
           "e5": "נתיב שגוי הוא הנחה על הסביבה", "r1_dates": "ה-AI הניח פורמט ברירת מחדל בלי לראות נתונים",
           "r2_range": "גבול הטווח הוא הנחה על הנתונים", "r3_dup_orders": "הנחה שהמזהה ייחודי",
           "r4_dup_suppliers": "הנחה שהמזהה ייחודי בטבלת הספקים"}
    for k, v in fam.items():
        assert family.get(k) in ("hallucination", "assumption"), f"family: ״{k}״ - כתבו hallucination או assumption"
        assert family[k] == v, f"family: ״{k}״ - חשבו שוב ({why[k]}). האם ה-AI המציא API שלא קיים?"
        assert isinstance(is_silent.get(k), bool), f"is_silent: ״{k}״ - כתבו True או False"
        assert is_silent[k] == (k.startswith("r")), f"is_silent: ״{k}״ - האם הייתה שגיאה שמפילה את התוכנית? (שקט = אין שגיאה)"
    return True

check_d() and print("✓ חלק ד': ההזיות הן e3 ו-e4, וארבעת באגי הדוח שקטים")

<div dir="rtl">

איזה סוג באגים מסוכן יותר לארגון - הזיות רועשות או הנחות שקטות על הנתונים? ובשביל כל אחד: מי מגלה אותו (המחשב, אתם, או רק מי שמשווה למקור חיצוני), ומה הכלי שתפס אותו במעבדה הזו?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

---

# חלק ה' - דיבוג ביחד עם AI: צעד אמיתי (5 דק')

## הרקע

כלל מההרצאה: **ה-AI יכול לעזור בכל צעד של הדיבוג, אבל את הניסוי ואת האימות עושים אתם.** כאן נתרגל את זה על **שגיאה אמיתית** (e4, `to_excel_fast`) ונעריך את האבחנה של הכלי **מול הכללים שלנו**, לא מול הרגשה.

## שלב 1 - בונים בקשת דיבוג טובה

בקשה טובה לוקחת ארבעה חלקים: **קוד, שגיאה, ציפייה, מה ניסיתי** - ובקשה מפורשת: **קודם הסבר סיבה, ורק אז תיקון.** ממשו `build_debug_request(code, error, expected, tried)` שמחזירה טקסט בצורה הזו (כל חלק בשורה/בלוק משלו):

```
Code:
<הקוד>
Error: <השגיאה>
Expected: <הציפייה>
Tried: <מה ניסיתי>
First explain the cause, then fix.
```

**רמז:** f-string עם `\n` בין החלקים, או `"\n".join([...])`.

</div>

In [ ]:
def build_debug_request(code, error, expected, tried):
    """Returns the 4-part debugging request (labels exactly: Code:, Error:, Expected:, Tried:)
    and a last line "First explain the cause, then fix."
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - חלק ה', שלב 1
def check_e1():
    r = build_debug_request("x = 1", "ValueError: boom", "x is 2", "nothing yet")
    for label in ("Code:", "Error:", "Expected:", "Tried:"):
        assert label in r, f"build_debug_request: חסר החלק ״{label}״"
    for part in ("x = 1", "ValueError: boom", "x is 2", "nothing yet"):
        assert part in r, f"build_debug_request: הערך ״{part}״ שהועבר לפונקציה לא מופיע בבקשה"
    assert "first explain the cause, then fix" in r.lower(), "build_debug_request: הוסיפו בסוף את השורה First explain the cause, then fix."
    assert r.index("Code:") < r.index("Error:") < r.index("Expected:") < r.index("Tried:"), "build_debug_request: הסדר - Code, Error, Expected, Tried"
    return True

check_e1() and print("✓ שלב 1: הבקשה בנויה בארבעה חלקים")

<div dir="rtl">

## שלב 2 - מדביקים לעוזר ה-AI

התא הבא מריץ את e4 ובונה את הבקשה **מהשורה האחרונה של ה-Traceback האמיתי** (מה שלמדתם לקרוא):

1. הריצו את התא והעתיקו את הבקשה שהודפסה.
2. פתחו את **Gemini של Colab** (כפתור ✨ בסרגל הצד, או לחצו **Explain error** שמופיע מתחת לשגיאה בתא) - או כל כלי צ'אט שאתם משתמשים בו.
3. הדביקו את הבקשה, וקראו **מה הכלי ענה**.

</div>

In [ ]:
tb4 = capture(E_CODES["e4.py"], "e4.py")
error_line = tb4.strip().splitlines()[-1]
request = build_debug_request(code=E_CODES["e4.py"],
                              error=error_line,
                              expected="the DataFrame is saved to an Excel file called out.xlsx",
                              tried="nothing yet")
print(request)

<div dir="rtl">

## שלב 3 - מנתחים את התשובה בכלים שבניתם

לא בודקים "האם התשובה נשמעת טוב". בודקים בקוד:

- `api_exists` (חלק א') - **האם השם שהכלי הציע באמת קיים?** (אם הוא המציא עוד שם - זו עוד הזיה.)
- `patch_smells(code)` - **האם הקוד שהציע מטלא?** (סימני טלאי מהלולאה בהרצאה: `try:`, `except`, `coerce`, `ignore` שמסתירים את הכשל.)

ממשו קודם את `patch_smells(code)`: מחזירה **רשימה** של הסימנים מבין `("try:", "except", "coerce", "ignore")` שמופיעים בטקסט `code`, **בסדר הזה**.

**רמז:** `for word in SMELLS: if word in code: ...` (המילה `in` על טקסט בודקת אם מילה מופיעה בו).

</div>

In [ ]:
SMELLS = ("try:", "except", "coerce", "ignore")

def patch_smells(code):
    """Returns the list of words from SMELLS that appear in the text `code` (in the order of SMELLS).
    e.g. patch_smells("try:\n    f()\nexcept:\n    pass") -> ["try:", "except"]
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - חלק ה', שלב 3 (patch_smells)
def check_e2():
    assert patch_smells('df.to_excel("out.xlsx")') == [], "patch_smells: קוד נקי - רשימה ריקה"
    assert patch_smells("try:\n    f()\nexcept:\n    pass") == ["try:", "except"], "patch_smells: try/except - שני סימנים, בסדר try: ואז except"
    assert patch_smells('pd.to_datetime(s, errors="coerce")') == ["coerce"], "patch_smells: coerce הוא סימן"
    assert patch_smells("warnings.filterwarnings('ignore')") == ["ignore"], "patch_smells: ignore הוא סימן"
    return True

check_e2() and print("✓ שלב 3: patch_smells עובדת")

<div dir="rtl">

עכשיו תעדו מה קיבלתם. **החליפו את הטקסטים שבסוגריים:**

- `ai_diagnosis` - 1-2 משפטים מההסבר של הכלי (העתיקו).
- `ai_method` - **שם המתודה** שהכלי הציע במקום `to_excel_fast` (למשל `to_...`).
- `ai_code` - הקוד שהכלי הציע לתיקון (העתיקו, גם אם זו שורה אחת).
- `ai_eval` - שני שדות **אתם** מחליטים (`True`/`False`): האם ההסבר בא **לפני** הקוד, והאם הכלי אמר במפורש שהמתודה **לא קיימת / מומצאת**.
- `decision` - `"adopted"` (אימצתי), `"rejected"` (דחיתי) או `"fixed"` (תיקנתי בעצמי).
- התא ידפיס גם את **הבדיקות שהכלים שלכם מחשבים** (`api_exists` ו-`patch_smells`) - אין מה למלא שם.

> הכלל: **אסור לאמץ** הצעה שהבדיקות שלכם לא אישרו (שם שלא קיים, או סימני טלאי).

</div>

In [ ]:
ai_diagnosis = "(הדביקו כאן את ההסבר של הכלי)"
ai_method = "(שם המתודה שהכלי הציע)"
ai_code = """(הדביקו כאן את הקוד שהכלי הציע)"""

ai_eval = {
    "cause_before_fix": None,        # True/False: ההסבר בא לפני הקוד?
    "named_hallucination": None,     # True/False: אמר במפורש שהמתודה מומצאת / לא קיימת?
}
decision = "?"      # "adopted" / "rejected" / "fixed"

def auto_eval():
    """The two checks done by YOUR tools (nothing to fill here)."""
    return {"no_patch": len(patch_smells(ai_code)) == 0,           # אין סימני טלאי בקוד?
            "method_exists": api_exists(pd.DataFrame, ai_method)}  # השם שהציע קיים?

print("הערכה שלי:", ai_eval)
print("הבדיקות שלכם:", auto_eval())
print("החלטה:", decision)

In [ ]:
# בדיקה - חלק ה', שלב 3 (התיעוד)
def check_e3():
    for name, value in (("ai_diagnosis", ai_diagnosis), ("ai_method", ai_method), ("ai_code", ai_code)):
        assert isinstance(value, str) and value.strip() and not value.strip().startswith("("), f"{name}: הדביקו את מה שהכלי ענה (במקום הטקסט שבסוגריים)"
    assert len(ai_diagnosis.strip()) >= 20, "ai_diagnosis: הדביקו לפחות משפט שלם מההסבר של הכלי"
    for key in ("cause_before_fix", "named_hallucination"):
        assert isinstance(ai_eval[key], bool), f"ai_eval: ״{key}״ - כתבו True או False"
    assert decision in ("adopted", "rejected", "fixed"), "decision: כתבו adopted, rejected או fixed"
    auto = auto_eval()
    if decision == "adopted":
        assert auto["method_exists"], "decision: אימצתם הצעה ש-api_exists לא אישרה (השם לא קיים) - דחו או תקנו"
        assert auto["no_patch"], "decision: אימצתם קוד עם סימני טלאי (patch_smells לא ריקה) - דחו או בקשו תיקון של השורש"
    return True

check_e3() and print("✓ שלב 3: האבחנה של הכלי הוערכה והוחלט מה עושים בה")

<div dir="rtl">

**(א)** הערכה לפי הכללים: האם הכלי הסביר סיבה **לפני** שתיקן? האם זיהה שמדובר בהזיה? האם ההצעה שלו עברה את בדיקות הקיום והטלאי שלכם? אילו חלקים מבקשת הדיבוג (קוד/שגיאה/ציפייה/מה ניסיתי) לדעתכם הוסיפו הכי הרבה לאיכות התשובה?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

**(ב)** נניח שהכלי הציע שם מתודה ש-`api_exists` החזירה עליו `False`, או עטף את הקוד ב-`try/except`. מה עושים לפי מה שלמדנו על **הזיות**, על **לולאת תיקונים** ועל **כלל העצירה**?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

---

# חלק ו' - מסקנה (3 דק')

חמישה משפטים, אחד לכל חלק. כל משפט בנוי כך: **מה מימשתם ומדדתם ← מה זה מראה על דיבוג של קוד AI ← מה עושים בעקבות זה.**

1. **Traceback והזיות:** _(כתבו כאן)_
2. **באג שקט:** _(כתבו כאן)_
3. **תיקון ובדיקת רגרסיה:** _(כתבו כאן)_
4. **טקסונומיה:** _(כתבו כאן)_
5. **דיבוג עם AI:** _(כתבו כאן)_

</div>

<div dir="rtl">

---

# בונוס

**1. בדיקת שפיות אחת לכולם.** כתבו `sanity_report(orders, suppliers)` שמחזירה מילון עם `nat`, `dup_orders`, `dup_suppliers` בעזרת הפונקציות שכבר בניתם, והריצו אותה **לפני** כל דוח. מה היה קורה לו הייתה רצה גם בדוח המקורי?

**2. טסט שלישי חזק יותר.** הוסיפו ל-`test_regression` בדיקה רביעית: מספר השורות בטבלה (`len(df)`) שווה ל-`CONTROL_ORDERS`. בדקו שהיא עדיין נכשלת על הגרסה "בלי התיקון `suppliers`" ומה בדיוק היא מוסיפה.

**3. באג בלי Traceback ל-AI.** הדביקו ל-Gemini את הקוד של `buggy_report` + "Actual TOTAL: 88526.81, Expected: 177597.89, Tried: nothing" **בלי לספר על התאריכים**. כמה מארבע הסיבות הוא מצא? (בדקו: האם הוא הציע try/except או קבוע? הפעילו את כלל העצירה אם כן.)

</div>

In [ ]:
# תא לבונוס 1

In [ ]:
# תא לבונוס 2

In [ ]:
# תא לבונוס 3

<div dir="rtl">

---

# בדיקה מסכמת - הריצו לפני ההגשה

התא הבא מריץ שוב את כל תאי הבדיקה. לפני ההגשה: **Runtime ← Restart session and run all**, וודאו שכל השורות כאן מסומנות `✓`.

</div>

In [ ]:
for name, check in [("א' - parse_traceback", check_a1),
                    ("א' - api_exists, package_exists", check_a2),
                    ("ב' - count_nat, count_dup_ids", check_b1),
                    ("ב' - find_bad_rows", check_b2),
                    ("ב' - מקרה מינימלי", check_b3),
                    ("ג' - parse_dates, in_q3", check_c1),
                    ("ג' - drop_dup_orders, drop_dup_suppliers", check_c2),
                    ("ג' - פנקס התיקונים", check_c3),
                    ("ג' - test_regression", check_c4),
                    ("ג' - תחקיר ו-commit", check_c5),
                    ("ד' - טקסונומיה", check_d),
                    ("ה' - build_debug_request", check_e1),
                    ("ה' - patch_smells", check_e2),
                    ("ה' - תיעוד העבודה עם ה-AI", check_e3)]:
    try:
        check()
        print(f"✓ {name}")
    except NotImplementedError:
        print(f"✗ {name}  ← עדיין לא מומש")
    except AssertionError as e:
        print(f"✗ {name}  ← {e}")